# 2. Add Two Numbers

**Difficulty:** Medium  
**Topics:** Linked List, Math, Recursion  
**Language:** Python 3  
**Problem:** [LeetCode — Add Two Numbers](https://leetcode.com/problems/add-two-numbers/)

Add two numbers represented by linked lists, one digit at a time. Start with a solution that buffers the digits in lists, then remove those buffers by traversing the linked lists directly. Both approaches use the same carry rule and return a newly allocated linked list.

Run the notebook from top to bottom. Only Python's standard library is needed. Edit the values in **Try your own input** to experiment; copy the final code cell into LeetCode to submit.

## 1. Problem

### Statement summary

Two nonempty singly linked lists encode nonnegative integers. Every node holds one decimal digit, with the least significant digit first. Return a linked list encoding the sum in that same order.

Neither input has unnecessary leading zeros in the represented number; the number zero is represented by a single node containing 0.

### Constraints

- Each input list has between 1 and 100 nodes, inclusive.
- Every node value is an integer from 0 through 9.
- A multi-node input ends with a nonzero digit, because its final node stores the most significant digit.

### Examples

**Example 1**

```text
Input: l1 = [2,4,3], l2 = [5,6,4]
Output: [7,0,8]
```

The inputs represent 342 and 465. Their sum is 807, represented least-significant digit first as 7 → 0 → 8.

**Example 2**

```text
Input: l1 = [0], l2 = [0]
Output: [0]
```

Zero plus zero produces a single zero node.

**Example 3**

```text
Input: l1 = [9,9,9,9,9,9,9], l2 = [9,9,9,9]
Output: [8,9,9,9,0,0,0,1]
```

The numbers are 9,999,999 and 9,999. Their sum is 10,009,998; a carry continues through several positions and creates an extra result node.

## 2. Understand the task

The bracket notation in the examples describes node values. The actual method receives two `ListNode` heads, not Python lists, and must return a `ListNode` head.

Reverse digit order is helpful: ordinary addition begins with the units position, which is already the head of each list. There is no need to reverse either input or convert an entire number into an integer.

At each digit position:

```text
total = first digit + second digit + incoming carry
result digit = total % 10
outgoing carry = total // 10
```

Treat a missing digit from a shorter list as 0. Continue while either input has a digit **or a carry remains**. Since the greatest possible total is 9 + 9 + 1 = 19, the carry is always 0 or 1.

The setup below supplies the `ListNode` type and conversion helpers for local execution. LeetCode supplies `ListNode` itself. Both implementations leave the input nodes unchanged; the returned nodes are newly allocated.

In [1]:
from typing import Optional

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def to_linked_list(digits):
    dummy = tail = ListNode()
    for digit in digits:
        tail.next = ListNode(digit)
        tail = tail.next
    return dummy.next


def to_digits(head):
    digits, visited = [], set()
    while head is not None:
        if id(head) in visited:
            raise ValueError("A linked list must not contain a cycle.")
        visited.add(id(head))
        digits.append(head.val)
        head = head.next
    return digits

EXAMPLE_L1 = [2, 4, 3]
EXAMPLE_L2 = [5, 6, 4]

## 3. Approach 1 — buffer the digits

### Idea

Copy the two linked lists into Python lists. Add their digits by index, storing each result digit in another Python list. Finally, build the result linked list from those digits. This separates linked-list traversal from arithmetic, at the cost of extra storage and passes.

### Algorithm

1. Traverse each input and collect its digits into a buffer.
2. Starting at index 0, add the corresponding digits and incoming carry. Use 0 when a buffer has no digit at that index.
3. Save `total % 10` and update `carry = total // 10`.
4. Continue until both buffers and the carry are exhausted.
5. Convert the buffered result digits into a linked list and return its head.

### Dry run

`l1 = [2,4,3]`, `l2 = [5,6,4]`. The copied buffers are `[2,4,3]` and `[5,6,4]`.

| Position | First digit | Second digit | Carry in | Total | Save digit | Carry out | Result buffer |
|---|---:|---:|---:|---:|---:|---:|---|
| Units | 2 | 5 | 0 | 7 | 7 | 0 | `[7]` |
| Tens | 4 | 6 | 0 | 10 | 0 | 1 | `[7,0]` |
| Hundreds | 3 | 4 | 1 | 8 | 8 | 0 | `[7,0,8]` |

There are three arithmetic iterations. No carry remains, so the result is 7 → 0 → 8.

### Why it works

After processing the first `k` digit positions, the result buffer holds the lowest `k` digits of the sum, and `carry` holds exactly what must be added at position `k`.

This is true before processing any digits. The identity `total = 10 * (total // 10) + total % 10` makes the next stored digit correct and preserves the remaining carry. Once both inputs and the carry are exhausted, all positions are correct. Copying the result digits into nodes preserves their order.

### Complexity

Let `m` and `n` be the input lengths and `L = max(m, n)`.

- **Time:** O(m + n). Copying the inputs, adding at most `L + 1` digit positions, and building the result are all linear.
- **Auxiliary space:** O(m + n), for the input and result buffers.
- **Returned output:** O(L) nodes, counted separately from auxiliary storage.

All arithmetic inside the digit loop uses values at most 19.

### Implementation

In [2]:
class Solution:
    def addTwoNumbers(self, l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
        first, second = [], []
        while l1 is not None:
            first.append(l1.val)
            l1 = l1.next
        while l2 is not None:
            second.append(l2.val)
            l2 = l2.next

        digits = []
        i = carry = 0
        while i < len(first) or i < len(second) or carry:
            a = first[i] if i < len(first) else 0
            b = second[i] if i < len(second) else 0
            total = a + b + carry
            digits.append(total % 10)
            carry = total // 10
            i += 1

        dummy = tail = ListNode()
        for digit in digits:
            tail.next = ListNode(digit)
            tail = tail.next
        return dummy.next

BufferedSolution = Solution
result = BufferedSolution().addTwoNumbers(to_linked_list(EXAMPLE_L1), to_linked_list(EXAMPLE_L2))
print(to_digits(result))

[7, 0, 8]


In [3]:
def trace_buffered(first, second):
    first, second = list(first), list(second)
    print(f"Digit buffers: {first} and {second}")
    digits = []
    i = carry = 0
    while i < len(first) or i < len(second) or carry:
        a = first[i] if i < len(first) else 0
        b = second[i] if i < len(second) else 0
        incoming = carry
        carry, digit = divmod(a + b + incoming, 10)
        digits.append(digit)
        print(f"Position {i}: {a} + {b} + carry {incoming} -> digit {digit}, carry {carry}; buffer={digits}")
        i += 1
    return digits

assert trace_buffered(EXAMPLE_L1, EXAMPLE_L2) == [7, 0, 8]

Digit buffers: [2, 4, 3] and [5, 6, 4]
Position 0: 2 + 5 + carry 0 -> digit 7, carry 0; buffer=[7]
Position 1: 4 + 6 + carry 0 -> digit 0, carry 1; buffer=[7, 0]
Position 2: 3 + 4 + carry 1 -> digit 8, carry 0; buffer=[7, 0, 8]


## 4. Approach 2 — add directly with a carry

### From buffered digits to a single traversal

The buffered solution only needs the current digit from each input. Keep a pointer to each current node instead, and append each computed digit directly to the result. This removes all three Python-list buffers while preserving the same arithmetic.

A dummy head makes appending the first result digit identical to appending later digits. Return `dummy.next`; the dummy node is not part of the answer.

### Algorithm

1. Create a dummy head, a result-tail pointer, and a zero carry.
2. While an input pointer or the carry remains, read the current digits, substituting 0 for a missing node.
3. Compute the digit and next carry using `divmod(total, 10)`.
4. Append one new result node and advance the available input pointers.
5. Return the node after the dummy head.

### Dry run

Use the same example: 2 → 4 → 3 plus 5 → 6 → 4.

| Visit | Current digits | Carry in | Total | Append | Carry out | Result so far |
|---|---|---:|---:|---:|---:|---|
| 1 | 2 and 5 | 0 | 7 | 7 | 0 | 7 |
| 2 | 4 and 6 | 0 | 10 | 0 | 1 | 7 → 0 |
| 3 | 3 and 4 | 1 | 8 | 8 | 0 | 7 → 0 → 8 |

Both input pointers are now `None`, and the carry is 0. The loop stops after three visits.

For a short supplementary carry example, `[9] + [1]` takes two visits: append 0 with carry 1, then append 1 even though both input lists have ended. This is why the loop condition includes `or carry`.

### Why it works

Before each iteration, the result chain contains the correct sum digits for every already-processed position, each input pointer identifies its first unprocessed digit, and `carry` is the amount carried into that position.

Initially, no positions have been processed and the carry is 0. Adding the two current digits and carry, then splitting the total with `divmod(total, 10)`, produces the correct next digit and next carry. Advancing the pointers preserves the invariant.

The loop stops only when both inputs and the carry are exhausted. Thus, every required digit has been appended and no carry has been lost. Returning `dummy.next` excludes the temporary sentinel.

### Complexity

- **Time:** O(max(m, n)), which is also O(m + n), with at most one visit per result digit.
- **Auxiliary space:** O(1), excluding the returned linked list. Only a few pointers and bounded integers are retained.
- **Returned output:** O(max(m, n)) nodes, with at most one extra node for a final carry.

The improvement is constant auxiliary space and fewer passes, not a better asymptotic time bound. Reading the input digits already requires linear time.

### Implementation

In [4]:
from typing import Optional

class Solution:
    def addTwoNumbers(self, l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
        dummy = tail = ListNode()
        carry = 0
        while l1 is not None or l2 is not None or carry:
            first = l1.val if l1 is not None else 0
            second = l2.val if l2 is not None else 0
            carry, digit = divmod(first + second + carry, 10)
            tail.next = ListNode(digit)
            tail = tail.next
            if l1 is not None:
                l1 = l1.next
            if l2 is not None:
                l2 = l2.next
        return dummy.next


CarrySolution = Solution
result = CarrySolution().addTwoNumbers(to_linked_list(EXAMPLE_L1), to_linked_list(EXAMPLE_L2))
print(to_digits(result))

[7, 0, 8]


In [5]:
def trace_carry(l1, l2):
    digits = []
    carry = visit = 0
    while l1 is not None or l2 is not None or carry:
        first = l1.val if l1 is not None else 0
        second = l2.val if l2 is not None else 0
        incoming = carry
        carry, digit = divmod(first + second + incoming, 10)
        digits.append(digit)
        visit += 1
        print(f"Visit {visit}: {first} + {second} + carry {incoming} -> digit {digit}, carry {carry}; result={digits}")
        l1 = l1.next if l1 is not None else None
        l2 = l2.next if l2 is not None else None
    return digits

assert trace_carry(to_linked_list(EXAMPLE_L1), to_linked_list(EXAMPLE_L2)) == [7, 0, 8]
print("Supplementary final-carry example:")
assert trace_carry(to_linked_list([9]), to_linked_list([1])) == [0, 1]

Visit 1: 2 + 5 + carry 0 -> digit 7, carry 0; result=[7]
Visit 2: 4 + 6 + carry 0 -> digit 0, carry 1; result=[7, 0]
Visit 3: 3 + 4 + carry 1 -> digit 8, carry 0; result=[7, 0, 8]
Supplementary final-carry example:
Visit 1: 9 + 1 + carry 0 -> digit 0, carry 1; result=[0]
Visit 2: 0 + 0 + carry 1 -> digit 1, carry 0; result=[0, 1]


## 5. Comparison and common mistakes

| Approach | Time | Auxiliary space | Main trade-off |
|---|---|---|---|
| Buffer the digits | O(m + n) | O(m + n) | Separates arithmetic from node traversal, but stores extra buffers |
| Direct addition with carry | O(m + n) | O(1) | Traverses the nodes directly and appends output immediately |

Use direct addition with carry for submission. Both algorithms allocate the required output nodes; O(1) auxiliary space does not mean that the result consumes no memory.

Common mistakes:

- Reading `[2,4,3]` as 243 instead of 342.
- Treating the Python-list example notation as the actual method input.
- Stopping when the shorter list ends, rather than using 0 for its missing digits.
- Dropping the last carry or failing to advance a pointer.
- Reversing the quotient and remainder from `divmod`: it returns `(carry, digit)`.
- Returning the dummy node instead of `dummy.next`.
- Using whole-number conversion without acknowledging fixed-width integer overflow in other languages or the cost of arbitrary-precision arithmetic.

A recursive version can apply the same carry rule, but adds a recursion stack. The iterative version keeps that additional storage constant. The local harness validates input constraints; its parsing and validation overhead is separate from the solution algorithms.

## 6. Test the solutions

Test all official examples and supplementary cases covering unequal lengths, zero, internal zeros, a final carry, and the maximum input length. For these implementations, also check that the input chains remain unchanged and that the result uses fresh nodes.

The test oracle reconstructs integers only for verification; the actual algorithms never convert the complete inputs to integers. Output traversal detects cycles so an invalid result cannot hang a test.

In [6]:
TEST_CASES = [
    ("Official example 1", [2, 4, 3], [5, 6, 4], [7, 0, 8]),
    ("Official example 2", [0], [0], [0]),
    ("Official example 3", [9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9], [8, 9, 9, 9, 0, 0, 0, 1]),
    ("Supplementary: final carry", [9], [1], [0, 1]),
    ("Supplementary: unequal lengths", [1], [9, 9], [0, 0, 1]),
    ("Supplementary: zero plus a number", [0], [3, 2, 1], [3, 2, 1]),
    ("Supplementary: internal zeros", [0, 1], [0, 9], [0, 0, 1]),
    ("Supplementary: no carry", [1, 2], [3, 4], [4, 6]),
    ("Supplementary: maximum length", [9] * 100, [1], [0] * 100 + [1]),
]


def numeric_value(digits):
    return sum(digit * 10**position for position, digit in enumerate(digits))


def node_ids(head):
    result = set()
    while head is not None:
        assert id(head) not in result, "Cycle detected."
        result.add(id(head))
        head = head.next
    return result


for name, first, second, expected in TEST_CASES:
    for implementation in (BufferedSolution, CarrySolution):
        l1, l2 = to_linked_list(first), to_linked_list(second)
        original_ids = node_ids(l1) | node_ids(l2)
        answer = implementation().addTwoNumbers(l1, l2)
        actual = to_digits(answer)
        assert actual == expected, (name, actual, expected)
        assert all(type(digit) is int and 0 <= digit <= 9 for digit in actual)
        assert len(actual) == 1 or actual[-1] != 0
        assert numeric_value(actual) == numeric_value(first) + numeric_value(second)
        assert to_digits(l1) == first and to_digits(l2) == second
        assert node_ids(answer).isdisjoint(original_ids)
    shown = expected if len(expected) <= 10 else f"{len(expected)} digits; last digit={expected[-1]}"
    print(f"PASS | {name}: {shown}")
print(f"Both implementations passed all {len(TEST_CASES)} cases.")

PASS | Official example 1: [7, 0, 8]
PASS | Official example 2: [0]
PASS | Official example 3: [8, 9, 9, 9, 0, 0, 0, 1]
PASS | Supplementary: final carry: [0, 1]
PASS | Supplementary: unequal lengths: [0, 0, 1]
PASS | Supplementary: zero plus a number: [3, 2, 1]
PASS | Supplementary: internal zeros: [0, 0, 1]
PASS | Supplementary: no carry: [4, 6]
PASS | Supplementary: maximum length: 101 digits; last digit=1
Both implementations passed all 9 cases.


## 7. Try your own input

Edit `l1_text`, `l2_text`, and `approach` in the next cell. Use JSON digit arrays in least-significant-first order. Choose `"carry"` or `"buffered"`.

The harness converts the arrays into linked lists, runs the chosen implementation, and displays the result as an array. It rejects empty inputs, invalid digits, excessive lengths, and unnecessary leading zeros. It uses JSON parsing, never `eval`, and does not block **Run All** with interactive prompts.

In [7]:
import json


def validate_digits(digits):
    if not isinstance(digits, list) or not 1 <= len(digits) <= 100:
        raise ValueError("Each input must contain 1 to 100 digits.")
    if any(type(value) is not int or not 0 <= value <= 9 for value in digits):
        raise ValueError("Each digit must be an integer from 0 through 9.")
    if len(digits) > 1 and digits[-1] == 0:
        raise ValueError("A multi-digit input must end with a nonzero digit.")


def run_manual(l1_text, l2_text, approach="carry"):
    try:
        first, second = json.loads(l1_text), json.loads(l2_text)
        validate_digits(first)
        validate_digits(second)
        implementations = {"carry": CarrySolution, "buffered": BufferedSolution}
        if approach not in implementations:
            raise ValueError("approach must be 'carry' or 'buffered'.")
        result = implementations[approach]().addTwoNumbers(to_linked_list(first), to_linked_list(second))
        digits = to_digits(result)
        print(f"{approach}: {first} + {second} -> {digits}")
        print(f"Number check: {numeric_value(first)} + {numeric_value(second)} = {numeric_value(digits)}")
        return digits
    except (ValueError, TypeError) as error:
        print(f"Input error: {error}")
        return None

# EDIT THESE VALUES, then rerun this cell.
l1_text = "[2, 4, 3]"
l2_text = "[5, 6, 4]"
approach = "carry"
manual_result = run_manual(l1_text, l2_text, approach)

carry: [2, 4, 3] + [5, 6, 4] -> [7, 0, 8]
Number check: 342 + 465 = 807


In [8]:
assert run_manual("[9]", "[1]", "buffered") == [0, 1]
assert run_manual("[]", "[1]") is None
assert run_manual("[10]", "[1]") is None
assert run_manual("[true]", "[1]") is None
assert run_manual("[1, 0]", "[1]") is None
assert run_manual("not JSON", "[1]") is None
assert run_manual(json.dumps([1] * 101), "[1]") is None
assert run_manual("[1]", "[1]", "unknown") is None
print("Manual-input harness checks passed.")

buffered: [9] + [1] -> [0, 1]
Number check: 9 + 1 = 10
Input error: Each input must contain 1 to 100 digits.
Input error: Each digit must be an integer from 0 through 9.
Input error: Each digit must be an integer from 0 through 9.
Input error: A multi-digit input must end with a nonzero digit.
Input error: Expecting value: line 1 column 1 (char 0)
Input error: Each input must contain 1 to 100 digits.
Input error: approach must be 'carry' or 'buffered'.
Manual-input harness checks passed.


## 8. Final submission

Copy only the next cell into LeetCode's Python 3 editor. LeetCode supplies the `ListNode` class, so the local node definition and conversion helpers are intentionally omitted. The method returns a linked-list head using the required `Solution.addTwoNumbers(self, l1, l2)` interface.

In [9]:
from typing import Optional

class Solution:
    def addTwoNumbers(self, l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
        dummy = tail = ListNode()
        carry = 0
        while l1 is not None or l2 is not None or carry:
            first = l1.val if l1 is not None else 0
            second = l2.val if l2 is not None else 0
            carry, digit = divmod(first + second + carry, 10)
            tail.next = ListNode(digit)
            tail = tail.next
            if l1 is not None:
                l1 = l1.next
            if l2 is not None:
                l2 = l2.next
        return dummy.next